TASK 1.1

In [0]:
from pyspark.sql.functions import *

video_path = "/Volumes/edtech_project/edtech_bronze/edtech_project/video_watch/"

# 1. Read Parquet
video_df = spark.read.parquet(video_path)

# 2. Convert watch_date properly
video_df = (video_df
    .withColumn("event_ts", to_timestamp("watch_date"))
    .withColumn("watch_date", to_date("watch_date"))  # overwrite directly
)

# 3. Add metadata columns
video_df = (video_df 
            .withColumn("_source_file", col("_metadata.file_path")) 
            .withColumn("_load_ts", current_timestamp()) 
            .withColumn("_schema_version", lit("v1")) )

# 4. Data Quality Checks
video_df = (video_df
    .filter(col("student_id").isNotNull())
    .filter(col("event_ts").isNotNull())
    .filter(col("event_ts") <= current_timestamp())
)

# 5. Write to Bronze table
(video_df.write
    .format("delta")
    .mode("append")
    .partitionBy("watch_date")
    .saveAsTable("video_events_bronze")
)

In [0]:
#volume alert
daily_counts = video_df.groupBy("watch_date").count()
daily_counts.filter((col("count") < 1_000_000) | (col("count") > 10_000_000)).show()

In [0]:

%sql
-- display the table
SELECT * 
FROM video_events_bronze
LIMIT 5;

In [0]:
%sql
--display partitons 
SHOW PARTITIONS video_events_bronze;